# 06 - Visualize Pre-MLP Concerto Room Features

This notebook visualizes a selected S3DIS room before the MLP translation head.

It uses the already extracted Concerto features, projects them from 896-D to RGB with PCA, and saves a figure with:
- raw RGB
- ground-truth labels
- Concerto features (PCA to RGB)

### 1. Setup Repo & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Project'
AREA_ID = 5
AREA_NAME = f'Area_{AREA_ID}'
FEATURES_NAME = f's3dis_area{AREA_ID}'
ROOM_NAME = 'Area_5_conferenceRoom_1'
DRIVE_FEATURES_DIR = f'{DRIVE_ROOT}/features/{FEATURES_NAME}'
LOCAL_FEATURES_ROOT = '/content/local_features'
LOCAL_FEATURES_DIR = f'{LOCAL_FEATURES_ROOT}/{FEATURES_NAME}'
RESULTS_DIR = f'{DRIVE_ROOT}/results/figures/concerto_pre_mlp'
BRANCH = 'dev/eval-demo'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

In [ ]:
%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull --no-edit origin {BRANCH}
!git restore notebooks/pyproject.toml scripts/visualize_concerto_pca.py
!git branch --show-current
!git log -1 --oneline

### 2. Wire Drive Paths and Copy Only the Selected Room Feature File Locally

This only deletes local symlinks and the temporary local room copy under `/content/local_features`. It does not delete anything from Drive.

In [ ]:
%cd {REPO_DIR}
!rm -f data features pretrained
!ln -sf {DRIVE_ROOT}/data ./data
!mkdir -p {LOCAL_FEATURES_ROOT}/{FEATURES_NAME}
!rm -f {LOCAL_FEATURES_DIR}/{ROOM_NAME}.npz
!cp {DRIVE_FEATURES_DIR}/{ROOM_NAME}.npz {LOCAL_FEATURES_DIR}/
!ln -sf {LOCAL_FEATURES_ROOT} ./features
!mkdir -p {RESULTS_DIR}
!echo Selected area: {AREA_NAME}
!echo Selected room: {ROOM_NAME}
!readlink -f ./features/{FEATURES_NAME}/{ROOM_NAME}.npz
!ls -lh {LOCAL_FEATURES_DIR}

In [ ]:
from pathlib import Path

feature_files = sorted(Path(DRIVE_FEATURES_DIR).glob('*.npz'))
print(f'Rooms found on Drive in {FEATURES_NAME}: {len(feature_files)}')
for path in feature_files[:30]:
    print(path.stem)

### 3. Setup Environment

In [ ]:
%cd {REPO_DIR}/notebooks
!uv python install 3.10.12
!uv sync --python 3.10.12

### 4. Generate Visualization

In [ ]:
%cd {REPO_DIR}

MAX_POINTS = 200000
POINT_SIZE = 1.5
SAVE_PNG = True
OUTPUT_BASE = f'{RESULTS_DIR}/{ROOM_NAME}_pca'

extra_args = '--save_png' if SAVE_PNG else ''
cmd = (
    'PYTHONPATH=/content/Deep_learning_project '
    'uv --project /content/Deep_learning_project/notebooks run python '
    '/content/Deep_learning_project/scripts/visualize_concerto_pca.py '
    f'--data_dir /content/Deep_learning_project/data/s3dis_processed '
    f'--features_dir /content/Deep_learning_project/features/{FEATURES_NAME} '
    f'--room {ROOM_NAME} '
    f'--max_points {MAX_POINTS} '
    f'--output {OUTPUT_BASE} '
    f'--point_size {POINT_SIZE} '
    f'{extra_args}'
)

print(cmd)
!{cmd}

### 5. Check Saved Outputs

In [ ]:
from pathlib import Path
from IPython.display import Image, display

html_path = Path(f'{OUTPUT_BASE}.html')
png_path = Path(f'{OUTPUT_BASE}.png')

print('HTML:', html_path, html_path.exists())
print('PNG :', png_path, png_path.exists())

if png_path.exists():
    display(Image(filename=str(png_path)))
else:
    print('PNG not generated. Open the HTML file from Drive if SAVE_PNG is False.')